<a href="https://colab.research.google.com/github/AktanM11/AI-OI/blob/main/DAY4_WEEK2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install qdrant-client

In [ ]:
pip install sentence-transformers

In [6]:
import uuid
import json
from qdrant_client import models
from qdrant_client import QdrantClient
from qdrant_client.http.models import Distance, VectorParams, PointStruct
from sentence_transformers import SentenceTransformer

In [ ]:
file_path = "policies.jsonl"
collection_name = "second_policies_collection"
chunk_size = 500
chunk_overlap = 50
model = SentenceTransformer('intfloat/multilingual-e5-large')
vector_dimension = model.get_sentence_embedding_dimension()

In [12]:
from google.colab import userdata
client = QdrantClient(url=userdata.get('QDRANT_URL'),
                      api_key=userdata.get('QDRANT_API_KEY'))

In [14]:
client.create_collection(
        collection_name=collection_name,
        vectors_config=VectorParams(size=vector_dimension, distance=Distance.COSINE),
    )

True

In [17]:
def recursive_chunking(text: str, chunk_size: int, overlap: int) -> list:
    if len(text) <= chunk_size:
        return [text] if text.strip() else []

    chunk = text[:chunk_size]
    slice_zone = chunk[-overlap:]
    last_space = slice_zone.rfind(" ")

    if last_space != -1:
        cut_index = (chunk_size - overlap) + last_space + 1
    else:
        cut_index = chunk_size

    final_chunk = text[:cut_index].strip()
    remaining_text = text[cut_index - overlap:]

    return [final_chunk] + recursive_chunking(remaining_text, chunk_size, overlap)

In [18]:
with open(file_path, "r", encoding="utf-8") as f:
    for line in f:
        if not line.strip():
            continue

        # Parse the JSON line
        data = json.loads(line)

        raw_text = data.get("content", "")
        # NEW: Extract metadata from the current line
        category = data.get("category", "Unknown")
        doc_type = data.get("doc_type", "Unknown")
        source_file = data.get("source_file", "Unknown")

        # Split text into chunks
        chunks = recursive_chunking(raw_text, chunk_size, chunk_overlap)

        points = []
        for chunk in chunks:
            if not chunk.strip():
                continue

            # Create vector (E5 model needs "passage: " prefix for text blocks)
            vector = model.encode(f"passage: {chunk}").tolist()

            # NEW: Put text AND metadata into the payload dictionary
            payload = {
                "page_content": chunk,
                "category": category,
                "doc_type": doc_type,
                "source_file": source_file
            }

            # Create the Qdrant point object
            point = PointStruct(
                id=str(uuid.uuid4()),
                vector=vector,
                payload=payload
            )
            points.append(point)

        # Upload points for this document immediately
        if points:
            client.upsert(collection_name=collection_name, points=points)

In [28]:
from qdrant_client.models import Filter, FieldCondition, MatchValue

In [39]:
from qdrant_client.models import PayloadSchemaType

In [40]:
client.create_payload_index(
    collection_name=collection_name,
    field_name="category",
    field_schema=PayloadSchemaType.KEYWORD, # Tells Qdrant this is text/string data
)

client.create_payload_index(
    collection_name=collection_name,
    field_name="source_file",
    field_schema=PayloadSchemaType.KEYWORD, # Tells Qdrant this is text/string data
)

UpdateResult(operation_id=89, status=<UpdateStatus.COMPLETED: 'completed'>)

In [43]:
def search_qdrant(query_text: str, search_filter: Filter = None):
    # E5 model needs "query: " prefix for questions/searches
    query_vector = model.encode(f"query: {query_text}").tolist()

    # Perform the search
    results = client.query_points(
        collection_name=collection_name,
        query=query_vector,
        query_filter=search_filter,
        limit=3
    )
    actual_points = results.points
    print(f"\n Results for query: '{query_text}'")
    if not results:
        print("No results found matching this filter.")
        return

    for idx, hit in enumerate(actual_points):
        print(f"\n[Result #{idx+1}] {hit.score:.4f}")
        print(f"-> Category: {hit.payload.get('category')}")
        print(f"-> Source File: {hit.payload.get('source_file')}")
        print(f"-> Text: {hit.payload.get('page_content')[:150]}...")

In [44]:
filter_2 = Filter(
    must_not=[
        FieldCondition(
            key="source_file",
            match=MatchValue(value="about.docx")
        )
    ]
)
search_qdrant(query_text="условия возврата устройств", search_filter=filter_2)


 Results for query: 'условия возврата устройств'

[Result #1] 0.8679
-> Category: Гарантия
-> Source File: warranty.docx
-> Text: В течение срока гарантии на заводской брак, который прописан в гарантийном талоне к товару; При наличии документов, подтверждающих факт и дату покупки...

[Result #2] 0.8552
-> Category: Гарантия
-> Source File: warranty.docx
-> Text: Условия возврата и обмена Если купленный товар в сети магазинов O!Store оказался с Существенным недостатком и невозможным для использования в соответс...

[Result #3] 0.8539
-> Category: Возврат
-> Source File: return_policy.docx
-> Text: , сохранен товарный вид и его неработоспособность подтверждается в момент обращения в O!Store для 4G-устройств компании или в Сервис-центр для телефон...


In [ ]:
{
 "product_id": "OSTORE_0002",
 "store_id": "osh_005",
 "product_name": "Apple iPhone 13 128 GB Starlight",
 "brand": "Apple",
 "store_name": "O!Store Ош Ареопаг",
 "store_city": "Ош",
 "store_region": "Ошская область",
 "store_address": "ул. Масалиева 10",
 "in_stock": true,
 "search_text": "Apple iPhone 13 128 GB Starlight Apple available at O!Store Ош Ареопаг Ош"
 }
